#### **El Escenario Comercial: ¿Dónde estamos parados?**
Superstore es una compañía líder en retail y comercio electrónico con operaciones a nivel nacional. La empresa comercializa tres grandes líneas de productos (Mobiliario, Suministros de Oficina y Tecnología) y atiende a tres perfiles de clientes claramente diferenciados: consumidores finales, oficinas corporativas y pequeñas empresas. En los últimos años, la compañía ha experimentado un crecimiento agresivo en su volumen de facturación, expandiendo su red logística a múltiples regiones y aplicando fuertes políticas promocionales para ganar cuota de mercado. Sin embargo, la alta dirección ha detectado una señal de alerta: el aumento en las ventas no se está traduciendo de forma lineal en un aumento de las ganancias netas.
#### **El Encargo de la Gerencia**
Ante esta situación, la Gerencia General de Superstore nos ha convocado como equipo de consultores analíticos para auditar la salud financiera y operativa de la organización. El mandato es claro: ir más allá de los reportes tradicionales de facturación y descubrir los factores ocultos que están afectando la rentabilidad.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sympy as sp
from scipy.optimize import curve_fit


pd.set_option("display.float_format", lambda x: "%.2f" % x)

url_dataset = "https://raw.githubusercontent.com/Pitiki10/TP-GRUPAL---SuperStore/main/data/superstore_dataset.csv"
df = pd.read_csv(url_dataset, encoding="latin1")

print("I. ESTRUCTURA Y TIPOS DE DATOS DEL DATASET")
print("======================================================================")
df.info()

print("\n" + "-"*70 + "\n")

print("II. ESTADÍSTICA DESCRIPTIVA DE VARIABLES NUMÉRICAS")
print("======================================================================")
print(df.describe())

I. ESTRUCTURA Y TIPOS DE DATOS DEL DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Ã¯Â»Â¿Row ID   8995 non-null   float64
 1   Order ID       8995 non-null   object 
 2   Order Date     8995 non-null   object 
 3   Ship Date      8995 non-null   object 
 4   Ship Mode      8995 non-null   object 
 5   Customer ID    8995 non-null   object 
 6   Customer Name  8995 non-null   object 
 7   Segment        8995 non-null   object 
 8   Country        8995 non-null   object 
 9   City           8995 non-null   object 
 10  State          8995 non-null   object 
 11  Postal Code    8995 non-null   float64
 12  Region         8995 non-null   object 
 13  Product ID     8995 non-null   object 
 14  Category       8995 non-null   object 
 15  Sub-Category   8995 non-null   object 
 16  Product Name   8995 non-null   object 
 17  Sales    

###**Limpieza y Manipulación de Datos (Data Wrangling)**
Luego de detectar anomalías estructurales en la codificación que comprometían la integridad del análisis cuantitativo, tales como la corrupción de nombres en columnas críticas, residuos de texto en variables financieras y la presencia de registros nulos extremos, se procede a corregir estas inconsistencias.

In [2]:
# Renombrar las columnas con errores de caracteres o formato
df.rename(columns={
    "Ã¯Â»Â¿Row ID": "Row ID",
    "Profit;": "Profit"
}, inplace=True)

# Mostrar estado inicial antes de eliminar nulos
print("I. CONTROL DE FILAS NULAS")
print("======================================================================")
print(f"Filas totales originales en el DataFrame: {len(df)}")

# Eliminar filas donde cualquiera de las columnas contenga un valor nulos
df.dropna(inplace=True)

print(f"Filas restantes después de eliminar registros nulos: {len(df)}")

print("\n" + "-"*70 + "\n")

# Evaluar si existen ID de fila repetidos (Duplicados)
print("II. EVALUACIÓN DE DUPLICADOS EN 'Row ID'")
print("======================================================================")
cantidad_duplicados = df["Row ID"].duplicated().sum()
print(f"Cantidad de IDs de fila repetidos detectados: {cantidad_duplicados}")

if cantidad_duplicados > 0:
    df.drop_duplicates(subset=["Row ID"], keep="first", inplace=True)
    print("▶️ Acción: Se eliminaron las filas duplicadas manteniendo la primera aparición.")
    print(f"Filas definitivas en el dataset: {len(df)}")
else:
    print("▶️ Acción: No se requieren eliminaciones. Cada fila posee un identificador único.")

# Transformacipon de la columna 'Profit' a tipo decimal (float)
df["Profit"] = df["Profit"].astype(str).str.rstrip(";")
df["Profit"] = pd.to_numeric(df["Profit"], errors="coerce")

print("\n" + "-"*70 + "\n")

print("III. VISTA GENERAL DEL DATASET LIMPIO")
print("======================================================================")
display(df.head())

I. CONTROL DE FILAS NULAS
Filas totales originales en el DataFrame: 9994
Filas restantes después de eliminar registros nulos: 8995

----------------------------------------------------------------------

II. EVALUACIÓN DE DUPLICADOS EN 'Row ID'
Cantidad de IDs de fila repetidos detectados: 0
▶️ Acción: No se requieren eliminaciones. Cada fila posee un identificador único.

----------------------------------------------------------------------

III. VISTA GENERAL DEL DATASET LIMPIO


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,2.00,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.00,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3.00,0.00,219.58
2,3.00,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.00,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2.00,0.00,6.87
4,5.00,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.00,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2.00,0.20,2.52
5,6.00,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032.00,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.86,7.00,0.00,14.17
6,7.00,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032.00,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4.00,0.00,1.97


### **🎯 Interrogante Principal de Alto Impacto**
**Pregunta**: ¿De qué manera la política de descuentos altera las funciones de ingreso y beneficio de Superstore, y cuál es el nivel óptimo de descuento que maximiza la ganancia neta antes de que la elasticidad de la demanda convierta las operaciones en pérdidas?

**Fundamentación Económica e Hipótesis**: Proponemos que la Ganancia Total (TB) no tiene un comportamiento lineal respecto al descuento, sino cóncavo (forma de U invertida). Estimamos que existe un umbral crítico de descuento donde la elasticidad-precio de la demanda deja de ser favorable; a partir de ese punto, la tasa de cambio del ingreso es menor a la tasa de cambio del costo operativo, provocando que la derivada del beneficio se vuelva negativa y destruya el margen unitario de la compañía. Al modelar analíticamente el Ingreso Total (TR) y el Costo Total (TC) en función del descuento, mediante derivadas y condiciones de primer y segundo orden, demostraremos que existe un punto de quiebre financiero

####🔄 **Preguntas Complementarias (Análisis de Mercado y Logística)**
Para robustecer el diagnóstico, aislaremos el problema principal cruzando variables de catálogo, geográficas y operativas bajo las siguientes ópticas:
1. Margen, Rentabilidad y Participación por Catálogo: ¿Cómo se distribuye la ganancia total y la participación porcentual del beneficio entre las distintas categorías, y qué subcategorías actúan como "anclas" financieras?. Se calculará el margen unitario y el aporte de cada sector al beneficio global para identificar si Tecnología subsidia las pérdidas ocultas de Mobiliario.

2. Análisis de Mercado ante Shocks Externos y Logística Regional: ¿Cómo reacciona la demanda de las categorías críticas en las regiones periféricas ante un shock externo en los costos logísticos, y cómo afecta esto a las tasas de cambio de los envíos?

### **Transformación Estratégica de Variables Núcleo**
**¿Qué se hace?** Se segmenta el dataset aislando exclusivamente las variables críticas para el motor de optimización financiera. Se calcula el Costo Total de cada transacción y el Precio Unitario real por producto. Adicionalmente, se crea una variable categórica para clasificar las transacciones en bloques comerciales estratégicos.

**¿Por qué se hace?** Para hallar el descuento óptimo mediante derivadas, requerimos estructurar la función de costos y el comportamiento del precio unitario. La clasificación por rangos es fundamental para la posterior fase de visualización y análisis de mercado; permitirá segmentar el comportamiento de la rentabilidad y demostrar visualmente en qué umbrales específicos la política promocional de la empresa genera valor económico o destruye el margen neto.

In [ ]:
# Creación de variables cuantitativas para el modelo matemático
df["Cost"] = df["Sales"] - df["Profit"]
df["Unit Price"] = df["Sales"] / df["Quantity"]

# Creación de rangos comerciales para la posterior visualización estratégica
# Definimos los límites (0% a 10%, 10% a 20%, y más de 20%)
limites = [-0.01, 0.07, 0.15, 0.30, 0.50, 1.00]
etiquetas = [
    "(0% - 7%)",
    "(8% - 15%)",
    "(16% - 30%)",
    "(31% - 50%)",
    "(> 50%)"
]

df["Discount Range"] = pd.cut(df["Discount"], bins=limites, labels=etiquetas)

# Auditoría visual de control del set unificado de optimización
print("VARIABLES NÚCLEO PARA EL ALGORITMO DE OPTIMIZACIÓN")
print("======================================================================")
columnas_nucleo = ["Unit Price", "Quantity", "Sales", "Cost", "Discount", "Discount Range", "Profit"]
display(df[columnas_nucleo].head())

print("\n" + "-"*70 + "\n")

# Agrupamos por los rangos y sumamos para auditar el volumen y la ganancia neta total
print("II. INTUICIÓN COMERCIAL: MÉTRICAS TOTALES POR RANGO DE DESCUENTO")
print("======================================================================")
sumatoria = df.groupby("Discount Range", observed=False)[["Sales", "Quantity", "Profit"]].sum()
display(sumatoria)

print("\n" + "-"*70 + "\n")

# Agrupamos por los rangos y calculamos la media para ver el comportamiento del cliente ante el precio
print("III. INTUICIÓN ECONÓMICA: PROMEDIOS UNITARIOS POR RANGO DE DESCUENTO")
print("======================================================================")
promedios = df.groupby("Discount Range", observed=False)[["Quantity", "Unit Price", "Profit", "Cost"]].mean()
promedios.columns = ["Avg Quantity", "Avg Unit Price", "Avg Profit", "Avg Cost"]
display(promedios)

VARIABLES NÚCLEO PARA EL ALGORITMO DE OPTIMIZACIÓN


,Unit Price,Quantity,Sales,Cost,Discount,Discount Range,Profit
0,130.98,2,261.96,220.05,0.00,(0% - 7%),41.91
1,243.98,3,731.94,512.36,0.00,(0% - 7%),219.58
2,7.31,2,14.62,7.75,0.00,(0% - 7%),6.87
3,191.52,5,957.58,1340.61,0.45,(31% - 50%),-383.03
4,11.18,2,22.37,19.85,0.20,(16% - 30%),2.52



----------------------------------------------------------------------

II. INTUICIÓN COMERCIAL: SUMAS TOTALES POR RANGO DE DESCUENTO


,Sales,Quantity,Profit
Discount Range,,,
(0% - 7%),1087908.47,18267,320987.60
(8% - 15%),81927.87,571,10448.17
(16% - 30%),867821.02,14509,79968.03
(31% - 50%),195314.76,1177,-48447.73
(> 50%),64228.74,3349,-76559.05



----------------------------------------------------------------------

III. INTUICIÓN ECONÓMICA: PROMEDIOS UNITARIOS POR RANGO DE DESCUENTO


,Avg Quantity,Avg Unit Price,Avg Profit,Avg Cost
Discount Range,,,,
(0% - 7%),3.81,58.91,66.90,159.84
(8% - 15%),3.91,145.80,71.56,489.59
(16% - 30%),3.74,61.79,20.59,202.85
(31% - 50%),3.80,159.27,-156.28,786.33
(> 50%),3.91,18.18,-89.44,164.47
